# 06 · Time series in pandas

Six hours of toy data, one of them missing, is enough to see what every time-series
method does. The real hourly file appears only at the end of each section.

**What's in here**
- DatetimeIndex: set, sort, check
- partial string indexing
- `.dt` on a column vs attributes on the index
- timezones: UTC vs local, DST
- `asfreq` vs `resample`, `label` / `closed`
- upsampling and filling
- detecting gaps
- `shift(n)` vs `shift(freq=)`, `rolling(n)` vs `rolling("nh")`
- the leakage pattern
- `diff` / `pct_change` on prices
- hour-of-day profiles, year-on-year pivot
- `between_time`, business days, calendar features
- checklist

In [1]:
import numpy as np
import pandas as pd

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 30)

## 1. A tiny series with a missing hour

Six timestamps, but 02:00 is absent. Keep this picture in mind for the whole notebook.

In [2]:
idx = pd.to_datetime(["2023-01-01 00:00", "2023-01-01 01:00", "2023-01-01 03:00",
                      "2023-01-01 04:00", "2023-01-01 05:00", "2023-01-01 06:00"])
s = pd.Series([10, 11, 13, 14, 15, 16], index=idx)
s

2023-01-01 00:00:00    10
2023-01-01 01:00:00    11
2023-01-01 03:00:00    13
2023-01-01 04:00:00    14
2023-01-01 05:00:00    15
2023-01-01 06:00:00    16
dtype: int64

## 2. DatetimeIndex: set it, sort it, check it

Three checks before any shift or rolling: is it a DatetimeIndex, is it sorted, is it unique.

In [3]:
print(type(s.index).__name__)
print("sorted :", s.index.is_monotonic_increasing)
print("unique :", s.index.is_unique)

DatetimeIndex
sorted : True
unique : True


From a DataFrame column: `set_index("time")` then `sort_index()`.

In [4]:
df = pd.DataFrame({"time": idx[::-1], "value": [16, 15, 14, 13, 11, 10]})   # deliberately reversed
df = df.set_index("time").sort_index()
df

,value
time,
2023-01-01 00:00:00,10
2023-01-01 01:00:00,11
2023-01-01 03:00:00,13
2023-01-01 04:00:00,14
2023-01-01 05:00:00,15
2023-01-01 06:00:00,16


## 3. Partial string indexing

With a DatetimeIndex you can select a day, a month or a year by string.

In [5]:
s.loc["2023-01-01"]              # the whole day

2023-01-01 00:00:00    10
2023-01-01 01:00:00    11
2023-01-01 03:00:00    13
2023-01-01 04:00:00    14
2023-01-01 05:00:00    15
2023-01-01 06:00:00    16
dtype: int64

In [6]:
s.loc["2023-01-01 03:00":"2023-01-01 04:00"]     # a label slice, inclusive

2023-01-01 03:00:00    13
2023-01-01 04:00:00    14
dtype: int64

## 4. `.dt` on a column vs attributes on the index

If the timestamps are a **column**, use `.dt.hour`. If they are the **index**, use `.hour` directly.

In [7]:
col = pd.Series(idx[:3])
print(col.dt.hour.tolist())          # column -> .dt
print(idx[:3].hour.tolist())         # index  -> attribute

[0, 1, 3]
[0, 1, 3]


In [8]:
pd.DataFrame({"hour": idx.hour, "dayofweek": idx.dayofweek, "date": idx.date}, index=idx).head(3)

,hour,dayofweek,date
2023-01-01 00:00:00,0,6,2023-01-01
2023-01-01 01:00:00,1,6,2023-01-01
2023-01-01 03:00:00,3,6,2023-01-01


## 5. Timezones

A naive timestamp has no zone. `tz_localize` **attaches** a zone (the clock reading stays
the same). `tz_convert` **changes** the zone (the instant stays the same, the clock changes).

In [9]:
t = pd.Series([1, 2], index=pd.to_datetime(["2023-07-01 12:00", "2023-07-01 13:00"]))
print(t.index.tz)
t_utc = t.tz_localize("UTC")
t_utc

None


2023-07-01 12:00:00+00:00    1
2023-07-01 13:00:00+00:00    2
dtype: int64

In [10]:
t_utc.tz_convert("Europe/London")       # same instants, London clock (BST = UTC+1)

2023-07-01 13:00:00+01:00    1
2023-07-01 14:00:00+01:00    2
dtype: int64

**Pitfall:** `tz_localize` on already-aware data raises; `tz_convert` on naive data raises.

In [11]:
try:
    t_utc.tz_localize("UTC")
except TypeError as e:
    print("TypeError:", e)
try:
    t.tz_convert("Europe/London")
except TypeError as e:
    print("TypeError:", e)

TypeError: Already tz-aware, use tz_convert to convert.
TypeError: Cannot convert tz-naive timestamps, use tz_localize to localize


**DST.** On 2023-03-26 the London clock jumped from 00:59 to 02:00. In UTC nothing happens.
Four UTC hours around the change, shown on both clocks:

In [12]:
utc = pd.date_range("2023-03-26 00:00", periods=4, freq="h", tz="UTC")
pd.DataFrame({"utc": utc, "london": utc.tz_convert("Europe/London")})

,utc,london
0,2023-03-26 00:00:00+00:00,2023-03-26 00:00:00+00:00
1,2023-03-26 01:00:00+00:00,2023-03-26 02:00:00+01:00
2,2023-03-26 02:00:00+00:00,2023-03-26 03:00:00+01:00
3,2023-03-26 03:00:00+00:00,2023-03-26 04:00:00+01:00


The London column skips 01:00. On 2023-10-29 the reverse happens: 01:00 London appears twice.

In [13]:
utc2 = pd.date_range("2023-10-29 00:00", periods=3, freq="h", tz="UTC")
pd.DataFrame({"utc": utc2, "london": utc2.tz_convert("Europe/London")})

,utc,london
0,2023-10-29 00:00:00+00:00,2023-10-29 01:00:00+01:00
1,2023-10-29 01:00:00+00:00,2023-10-29 01:00:00+00:00
2,2023-10-29 02:00:00+00:00,2023-10-29 02:00:00+00:00


This is why you **store UTC** (unique, monotonic) and derive the local hour only as a feature.
Localising naive *local* timestamps at the change is ambiguous; you must tell pandas what to do.

In [14]:
local_naive = pd.to_datetime(["2023-10-29 01:00", "2023-10-29 01:00"])
try:
    local_naive.tz_localize("Europe/London")
except Exception as e:
    print(type(e).__name__, ":", str(e)[:60])
local_naive.tz_localize("Europe/London", ambiguous=[True, False])   # first is BST, second is GMT

AmbiguousTimeError : Cannot infer dst time from 2023-10-29 01:00:00, try using th


DatetimeIndex(['2023-10-29 01:00:00+01:00', '2023-10-29 01:00:00+00:00'], dtype='datetime64[ns, Europe/London]', freq=None)

## 6. `asfreq` vs `resample`

`asfreq("h")` just puts the series on a regular grid: existing values stay, the missing
02:00 appears as NaN. No aggregation.

In [15]:
s.asfreq("h")

2023-01-01 00:00:00    10.0
2023-01-01 01:00:00    11.0
2023-01-01 02:00:00     NaN
2023-01-01 03:00:00    13.0
2023-01-01 04:00:00    14.0
2023-01-01 05:00:00    15.0
2023-01-01 06:00:00    16.0
Freq: h, dtype: float64

`resample("2h")` groups into 2-hour bins and needs an aggregation.

In [16]:
s.resample("2h").mean()

2023-01-01 00:00:00    10.5
2023-01-01 02:00:00    13.0
2023-01-01 04:00:00    14.5
2023-01-01 06:00:00    16.0
Freq: 2h, dtype: float64

- 00:00 bin = mean(10, 11) = 10.5
- 02:00 bin = 13 (only 03:00 exists)
- 04:00 bin = mean(14, 15) = 14.5
- 06:00 bin = 16

`sum` for energy, `mean` for power/price, `agg` with a dict for a DataFrame.

In [17]:
df = pd.DataFrame({"mwh": s, "price": [50, 52, 60, 58, 55, 54]})
df.resample("2h").agg({"mwh": "sum", "price": "mean"})

,mwh,price
2023-01-01 00:00:00,21,51.0
2023-01-01 02:00:00,13,60.0
2023-01-01 04:00:00,29,56.5
2023-01-01 06:00:00,16,54.0


## 7. `label` and `closed`: which timestamp names the bar?

Same 2-hour bins, `sum` this time, four combinations. `closed` says which edge of the bin
is included; `label` says which edge names the bin.

In [18]:
s.resample("2h", label="left", closed="left").sum()      # the default

2023-01-01 00:00:00    21
2023-01-01 02:00:00    13
2023-01-01 04:00:00    29
2023-01-01 06:00:00    16
Freq: 2h, dtype: int64

Bin [00:00, 02:00) holds 00:00 and 01:00 → 10 + 11 = 21, and is labelled 00:00 (its left edge).

In [19]:
s.resample("2h", label="right", closed="left").sum()

2023-01-01 02:00:00    21
2023-01-01 04:00:00    13
2023-01-01 06:00:00    29
2023-01-01 08:00:00    16
Freq: 2h, dtype: int64

Same bins, but each is now labelled by its **right** edge: the 21 sits at 02:00.

In [20]:
s.resample("2h", label="left", closed="right").sum()

2022-12-31 22:00:00    10
2023-01-01 00:00:00    11
2023-01-01 02:00:00    27
2023-01-01 04:00:00    31
Freq: 2h, dtype: int64

Bins are now (22:00, 00:00], (00:00, 02:00], … so 00:00 falls in the first bin on its own (10)
(00:00, 02:00] holds only 01:00 → 11; (02:00, 04:00] holds 03:00 and 04:00 → 27.

In [21]:
s.resample("2h", label="right", closed="right").sum()

2023-01-01 00:00:00    10
2023-01-01 02:00:00    11
2023-01-01 04:00:00    27
2023-01-01 06:00:00    31
Freq: 2h, dtype: int64

Default is `label="left", closed="left"`: the bar stamped 00:00 contains 00:00 and 01:00.
With `closed="right"` the bar (00:00, 02:00] contains 01:00 and 02:00 instead.

**Interview check:** *"The 4-hour bar at 08:00 — does it contain 08:00–12:00 or 04:00–08:00?"*
— Depends on `label`. Print one bar and check before you trust any resampled feature.

## 8. Upsampling and filling

Going from hourly to 30-minute creates empty slots. How you fill them matters.

In [22]:
h = pd.Series([10, 12], index=pd.to_datetime(["2023-01-01 00:00", "2023-01-01 01:00"]))
pd.DataFrame({
    "asfreq": h.asfreq("30min"),
    "ffill": h.asfreq("30min").ffill(),
    "interpolate": h.asfreq("30min").interpolate(),
})

,asfreq,ffill,interpolate
2023-01-01 00:00:00,10.0,10.0,10.0
2023-01-01 00:30:00,NaN,10.0,11.0
2023-01-01 01:00:00,12.0,12.0,12.0


`ffill` repeats the last known value (safe: only uses the past). `interpolate` uses the
**next** value too (00:30 = 11 needs to know 01:00 = 12), which is look-ahead in a
forecasting setting.

## 9. Detecting gaps

Three ways to find the missing 02:00.

In [23]:
s.index.to_series().diff().value_counts()

0 days 01:00:00    4
0 days 02:00:00    1
Name: count, dtype: int64

Five steps of 1 hour and one step of 2 hours → one missing hour.

In [24]:
full = pd.date_range(s.index.min(), s.index.max(), freq="h")
missing = full.difference(s.index)
missing

DatetimeIndex(['2023-01-01 02:00:00'], dtype='datetime64[ns]', freq='h')

In [25]:
print(pd.infer_freq(s.index))                 # None: not regular
print(pd.infer_freq(s.asfreq("h").index))     # 'h' once on the grid

None
h


## 10. `shift(n)` vs `shift(freq=)`: rows are not hours

`shift(1)` moves values down **one row**. Across the gap, 03:00 receives the 01:00 value —
that is a 2-hour-old value pretending to be 1 hour old.

`shift(freq="1h")` moves the **timestamps** by one hour instead; the value that was at
01:00 is now labelled 02:00, and 03:00 gets nothing (NaN after alignment).

In [26]:
pd.DataFrame({
    "s": s,
    "shift(1)": s.shift(1),
    "shift(freq='1h')": s.shift(freq="1h"),
})

,s,shift(1),shift(freq='1h')
2023-01-01 00:00:00,10.0,NaN,NaN
2023-01-01 01:00:00,11.0,10.0,10.0
2023-01-01 02:00:00,NaN,NaN,11.0
2023-01-01 03:00:00,13.0,11.0,NaN
2023-01-01 04:00:00,14.0,13.0,13.0
2023-01-01 05:00:00,15.0,14.0,14.0
2023-01-01 06:00:00,16.0,15.0,15.0
2023-01-01 07:00:00,NaN,NaN,16.0


Row 03:00: `shift(1)` says 11 (the 01:00 value), `shift(freq="1h")` says NaN (nothing was at 02:00).
The table also grew rows 02:00 and 07:00: those are timestamps the `freq` version created.
On a complete grid the two agree; with gaps only the `freq` version is honest.

## 11. `rolling(n)` vs `rolling("nh")`

`rolling(3)` = last 3 **rows**. `rolling("3h")` = every row inside the last 3 **hours**.

In [27]:
pd.DataFrame({
    "s": s,
    "rolling(3).mean()": s.rolling(3).mean(),
    "rolling('3h').mean()": s.rolling("3h").mean(),
})

,s,rolling(3).mean(),rolling('3h').mean()
2023-01-01 00:00:00,10,NaN,10.0
2023-01-01 01:00:00,11,NaN,10.5
2023-01-01 03:00:00,13,11.333333,12.0
2023-01-01 04:00:00,14,12.666667,13.5
2023-01-01 05:00:00,15,14.000000,14.0
2023-01-01 06:00:00,16,15.000000,15.0


- Row 03:00: `rolling(3)` = mean(10, 11, 13) — reaches back to 00:00, three rows.
  `rolling("3h")` = mean(11, 13) — only 01:00 and 03:00 are within 3 hours.
- The time window needs no warm-up (first row already has a value), the row window needs 3 rows.

`min_periods` controls how many observations a window needs before it reports a value.

In [28]:
pd.DataFrame({"rolling(3)": s.rolling(3).mean(), "min_periods=1": s.rolling(3, min_periods=1).mean()})

,rolling(3),min_periods=1
2023-01-01 00:00:00,NaN,10.000000
2023-01-01 01:00:00,NaN,10.500000
2023-01-01 03:00:00,11.333333,11.333333
2023-01-01 04:00:00,12.666667,12.666667
2023-01-01 05:00:00,14.000000,14.000000
2023-01-01 06:00:00,15.000000,15.000000


Other rolling statistics work the same way.

In [29]:
pd.DataFrame({"std": s.rolling(3).std(), "max": s.rolling(3).max(), "expanding_mean": s.expanding().mean(), "ewm": s.ewm(halflife=2).mean()}).round(2)

,std,max,expanding_mean,ewm
2023-01-01 00:00:00,NaN,NaN,10.00,10.00
2023-01-01 01:00:00,NaN,NaN,10.50,10.59
2023-01-01 03:00:00,1.53,13.0,11.33,11.68
2023-01-01 04:00:00,1.53,14.0,12.00,12.59
2023-01-01 05:00:00,1.00,15.0,12.60,13.44
2023-01-01 06:00:00,1.00,16.0,13.17,14.30


## **Pitfall:** the leakage pattern — `rolling(3).mean()` includes *now*

`rolling(3).mean()` at row 04:00 averages the three rows **ending at 04:00**, so the 04:00
value is inside its own feature. If 04:00 is what you are predicting, the feature already
contains the answer. `shift(1)` first, so the window ends at the previous row.

In [30]:
pd.DataFrame({
    "s": s,
    "rolling(3).mean()": s.rolling(3).mean(),
    "shift(1).rolling(3).mean()": s.shift(1).rolling(3).mean(),
})

,s,rolling(3).mean(),shift(1).rolling(3).mean()
2023-01-01 00:00:00,10,NaN,NaN
2023-01-01 01:00:00,11,NaN,NaN
2023-01-01 03:00:00,13,11.333333,NaN
2023-01-01 04:00:00,14,12.666667,11.333333
2023-01-01 05:00:00,15,14.000000,12.666667
2023-01-01 06:00:00,16,15.000000,14.000000


Row 04:00: leaked = mean(11, 13, 14) = 12.67 (rows 01:00, 03:00, 04:00 — includes itself);
honest = mean(10, 11, 13) = 11.33 (the three rows **before** 04:00).

**Interview check:** *"Which of these columns would be available at time t?"* — Only the
shifted one. The shorter the window, the more the leaked version is just the target itself.

How much it flatters you on real data: correlation of consumption with each feature.

In [31]:
hourly = pd.read_csv("../data/hourly_power_clean.csv", parse_dates=["time"]).set_index("time")
c = hourly["consumption_mwh"]
for w in [2, 6, 24]:
    leaked = c.rolling(w).mean().corr(c)
    honest = c.shift(1).rolling(w).mean().corr(c)
    print(f"window {w:2d}:  leaked corr {leaked:.3f}   honest corr {honest:.3f}")

window  2:  leaked corr 0.984   honest corr 0.873
window  6:  leaked corr 0.749   honest corr 0.573


window 24:  leaked corr 0.495   honest corr 0.488


## 12. `diff` and `pct_change` on prices that cross zero

`diff` is always meaningful. `pct_change` divides by the previous value: near zero it
explodes, and across a sign change it lies.

In [32]:
price = pd.Series([50.0, 5.0, -5.0, 10.0])
pd.DataFrame({"price": price, "diff": price.diff(), "pct_change": price.pct_change().round(2)})

,price,diff,pct_change
0,50.0,NaN,NaN
1,5.0,-45.0,-0.9
2,-5.0,-10.0,-2.0
3,10.0,15.0,-3.0


From −5 to 10 is a rise, but `pct_change` says −300%. For power prices use `diff`.

## 13. Hour-of-day and day-of-week profiles

Group by a time component of the index and average.

In [33]:
tiny = pd.Series([1, 2, 3, 4], index=pd.to_datetime(["2023-01-02 08:00", "2023-01-02 18:00", "2023-01-03 08:00", "2023-01-03 18:00"]))
tiny.groupby(tiny.index.hour).mean()

8     2.0
18    3.0
dtype: float64

08:00 → mean(1, 3) = 2; 18:00 → mean(2, 4) = 3. On the real data:

In [34]:
profile = c.groupby([c.index.dayofweek.rename("dow"), c.index.hour.rename("hour")]).mean().unstack("hour")
profile.round(0).iloc[:, [0, 8, 12, 18]]

hour,0,8,12,18
dow,,,,
0,26013.0,32338.0,31672.0,36061.0
1,26019.0,32179.0,31377.0,35814.0
2,26047.0,32115.0,31431.0,36026.0
3,26097.0,32297.0,31438.0,36027.0
4,25913.0,32159.0,31497.0,35910.0
5,23937.0,30187.0,29366.0,33889.0
6,24014.0,30094.0,29374.0,33839.0


## 14. Year-on-year comparison with a pivot

Rows = day of year, columns = year. Two years side by side.

In [35]:
daily = c.resample("D").mean()
yoy = pd.DataFrame({"value": daily, "year": daily.index.year, "doy": daily.index.dayofyear})
yoy.pivot(index="doy", columns="year", values="value").round(0).head(3)

year,2022,2023
doy,,
1,30587.0,30072.0
2,30375.0,32352.0
3,31856.0,31552.0


## 15. `between_time` / `at_time`

Select by clock time regardless of the date.

In [36]:
s.between_time("03:00", "05:00")

2023-01-01 03:00:00    13
2023-01-01 04:00:00    14
2023-01-01 05:00:00    15
dtype: int64

In [37]:
c.at_time("18:00").head(3)

time
2022-01-01 18:00:00+00:00    35038.8
2022-01-02 18:00:00+00:00    35773.4
2022-01-03 18:00:00+00:00    37022.6
Name: consumption_mwh, dtype: float64

## 16. Business days

`bdate_range` skips weekends; pass `holidays=` with `freq="C"` to skip holidays too.

In [38]:
pd.bdate_range("2023-12-22", "2023-12-28")

DatetimeIndex(['2023-12-22', '2023-12-25', '2023-12-26', '2023-12-27', '2023-12-28'], dtype='datetime64[ns]', freq='B')

In [39]:
pd.bdate_range("2023-12-22", "2023-12-28", freq="C", holidays=["2023-12-25", "2023-12-26"])

DatetimeIndex(['2023-12-22', '2023-12-27', '2023-12-28'], dtype='datetime64[ns]', freq='C')

## 17. A calendar-feature block you can paste anywhere

Each line adds one column from the (local) timestamp index.

In [40]:
feat = pd.DataFrame(index=s.index)
feat["hour"] = feat.index.hour
feat["dow"] = feat.index.dayofweek
feat["month"] = feat.index.month
feat["is_weekend"] = (feat.index.dayofweek >= 5).astype(int)
feat["is_holiday"] = feat.index.normalize().isin(pd.to_datetime(["2023-01-01"])).astype(int)
feat

,hour,dow,month,is_weekend,is_holiday
2023-01-01 00:00:00,0,6,1,1,1
2023-01-01 01:00:00,1,6,1,1,1
2023-01-01 03:00:00,3,6,1,1,1
2023-01-01 04:00:00,4,6,1,1,1
2023-01-01 05:00:00,5,6,1,1,1
2023-01-01 06:00:00,6,6,1,1,1


## Time-series checklist

1. Timestamps parsed (`datetime64`), tz-aware, stored in UTC.
2. Index sorted and unique.
3. Gaps found (`diff().value_counts()` or `date_range` difference) and decided on.
4. `shift` / `rolling` on a complete grid, or use the `freq=` / `"nh"` forms.
5. Rolling features shifted by 1 before use as features.
6. `resample`: `sum` for energy, `mean` for power; check `label` / `closed` on one bar.
7. Local clock only for features (hour, weekend, holidays), never for the index.
8. `diff`, not `pct_change`, for prices that can be zero or negative.